# Hands-On with Docling for IBM watsonx
## Developer Notebook (Part 2B)

**Use case:** You are a compliance analyst at *Meridian Financial Group*. Your task is to process a bundle of regulatory documents (PDF, DOCX, EPUB) and extract the key metrics needed to fill a standardised Compliance Assessment Form.

**This notebook covers:**
1. Connect to the Docling for IBM watsonx managed service
2. Convert documents
3. Inspect document structure
4. Export to DocLang and query with `dclq`
5. Chunk documents for RAG
6. RAG pipeline with LangChain + Milvus + watsonx.ai
7. Information extraction with a Pydantic schema
8. Putting it all together — fill the Compliance Assessment Form
9. *(Optional)* Edit the form document with Docling Agent

---
## Setup

In [ ]:
# Install dependencies
# Run this cell once, then restart the kernel if prompted.
! uv pip install -q \
    "docling-slim[service-client,feat-chunking]" \
    langchain-docling langchain-core langchain-ibm langchain-classic langchain-openai \
    "langchain-milvus" "pymilvus[milvus_lite]" \
    python-dotenv dclq rich

In [ ]:
import logging
import os

from dotenv import load_dotenv

logging.basicConfig(level=logging.ERROR)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Load DOCLING_SERVICE_URL, DOCLING_SERVICE_API_KEY, WX_API_KEY, WX_PROJECT_ID from .env
load_dotenv()

DOCLING_SERVICE_URL = os.environ["DOCLING_SERVICE_URL"]
DOCLING_SERVICE_API_KEY = os.environ["DOCLING_SERVICE_API_KEY"]

print("Service URL configured:", DOCLING_SERVICE_URL[:40] + "...")

---
## Section 1 — Connect to the Managed Service

`DoclingServiceClient` is the Python entry point for **Docling for IBM watsonx**.
Its call shape mirrors the local `DocumentConverter` — swap the URL and API key and
your code is portable between local and managed execution.

In [ ]:
from docling.service_client import DoclingServiceClient

client = DoclingServiceClient(
    url=DOCLING_SERVICE_URL,
    api_key=DOCLING_SERVICE_API_KEY,
)
print("Client ready.")

---
## Section 2 — Convert Documents

We convert the three documents that make up Meridian's Q3 compliance bundle.
All heavy work (OCR, table structure detection, layout analysis) happens on the
managed service — **no local ML models or GPU required**.

In [ ]:
# Document sources — local paths or public URLs
# Replace with the actual sample documents provided for this lab.
SOURCES = [
    "data/meridian_risk_disclosure.pdf",
    "data/q3_audit_summary.docx",
    "data/regulatory_guidelines_2025.epub",
]

In [ ]:
# Convert one document
result = client.convert(source=SOURCES[0])
doc = result.document

print(f"Document: {doc.name}")
print(f"Status:   {result.status}")
print("\n--- Markdown preview (first 1000 chars) ---\n")
print(doc.export_to_markdown()[:1000])

In [ ]:
# Convert all three documents concurrently
all_results = list(client.convert_all(source=SOURCES, max_concurrency=3))

docs = {}
for r in all_results:
    name = r.input.file.name if r.input and r.input.file else str(r.input)
    print(f"  {name:45s}  {r.status.value}")
    if r.document:
        docs[name] = r.document

print(f"\nSuccessfully converted: {len(docs)}/{len(SOURCES)} documents")

---
## Section 3 — Inspect Document Structure

A `DoclingDocument` is a rich tree: headings nest sections, which contain
paragraphs, tables, figures, lists, and more. You can iterate the tree in Python
to understand the document without sending it to an LLM.

In [ ]:
# Print the heading outline of the first document
primary_doc = docs.get("meridian_risk_disclosure.pdf") or list(docs.values())[0]

print("=== Heading outline ===")
for item, level in primary_doc.iterate_items():
    if hasattr(item, "label") and item.label.name == "SECTION_HEADER":
        indent = "  " * (level - 1)
        print(f"{indent}{'#' * level} {item.text}")

In [ ]:
# Inspect tables
print(f"Tables found: {len(primary_doc.tables)}")
for i, table in enumerate(primary_doc.tables):
    print(f"\n--- Table {i + 1} ---")
    print(table.export_to_markdown())

In [ ]:
# Inspect pictures / figures
print(f"Pictures found: {len(primary_doc.pictures)}")
for pic in primary_doc.pictures:
    caption = pic.caption_text(primary_doc)
    if caption:
        print(f"  Caption: {caption[:120]}")

---
## Section 4 — Export to DocLang and Query with `dclq`

**DocLang** (`.dclg` / `.dclx`) is an open XML-based ISO standard for structured
documents. Every heading, paragraph, table cell, and footnote has an XPath address.

`dclq` lets you grep, list, outline, and XPath-query DocLang files — **no LLM needed**.

In [ ]:
import os
from pathlib import Path

output_dir = Path("output")
output_dir.mkdir(exist_ok=True)

# Export the primary document to DocLang (.dclx bundle)
dclx_path = output_dir / "meridian_risk_disclosure.dclx"
primary_doc.save_as_doclang(dclx_path)   # produces a .dclx ZIP bundle
print(f"Saved DocLang file: {dclx_path}")

In [ ]:
# Inspect the document structure
! dclq inspect {dclx_path}

In [ ]:
# Print the heading outline with XPath addresses
! dclq outline {dclx_path}

In [ ]:
# Search for risk-related content
! dclq grep -i 'risk exposure' {dclx_path}

In [ ]:
# Retrieve a specific section by XPath (replace the XPath with one from the outline above)
# Example: dclq show doc.dclx '/heading[5]' --section
! dclq show {dclx_path} '/heading[2]' --section --max-chars 800

In [ ]:
# List all table cells on page 1
! dclq list {dclx_path} --type table_cell --page 1

> 📖 `dclq` is experimental. Full command reference: https://github.com/docling-project/docling-core/tree/main/packages/dclq

---
## Section 5 — Chunk Documents for RAG

The managed service can chunk a document in a single call — **no local models needed**.
`ChunkerKind.HYBRID` splits by heading hierarchy first, then by token count.
Each chunk retains its heading breadcrumb and source page number.

In [ ]:
from docling.service_client import ChunkerKind

chunk_response = client.chunk(
    source=SOURCES[0],
    chunker=ChunkerKind.HYBRID,
)

print(f"{len(chunk_response.chunks)} chunks from {len(chunk_response.documents)} document(s)")
print()
for chunk in chunk_response.chunks[:3]:
    print(f"--- page {chunk.meta.origin.page_no if chunk.meta and chunk.meta.origin else '?'} ---")
    print(chunk.text[:300])
    print()

---
## Section 6 — RAG Pipeline

### !!! REMOVE THIS SECTION OR REPLACE WITH CHUNKLESS RAG !!! ###

We build a retrieval-augmented generation pipeline using:
- **DoclingLoader** (LangChain) to convert and chunk in one step
- **Milvus Lite** as the local vector store
- **watsonx.ai** (`ibm/granite-4-h-small`) as the LLM for generation

The service handles all conversion — the notebook only orchestrates the pipeline.

In [ ]:
# Ingestion — convert, chunk, embed, and store
from pathlib import Path
from tempfile import mkdtemp

from langchain_docling import DoclingLoader
from langchain_docling.loader import ExportType
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_milvus import Milvus

EMBEDDING_MODEL = "ibm-granite/granite-embedding-30m-english"

# DoclingLoader uses the service-client under the hood when env vars are set
loader = DoclingLoader(
    file_path=SOURCES,
    export_type=ExportType.DOC_CHUNKS,
)
chunks = loader.load()
print(f"Loaded {len(chunks)} chunks from {len(SOURCES)} documents")

In [ ]:
# Build vector store
milvus_uri = str(Path(mkdtemp()) / "compliance.db")

vector_store = Milvus.from_documents(
    documents=chunks,
    embedding=HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL),
    collection_name="compliance_demo",
    connection_args={"uri": milvus_uri},
    index_params={"index_type": "FLAT"},
    drop_old=True,
)
retriever = vector_store.as_retriever(search_kwargs={"k": 4})
print("Vector store ready.")

In [ ]:
# LLM — watsonx.ai
from langchain_ibm import ChatWatsonx

WX_API_KEY = os.environ.get("WX_API_KEY")
WX_PROJECT_ID = os.environ.get("WX_PROJECT_ID")
if not WX_API_KEY or not WX_PROJECT_ID:
    raise RuntimeError(
        "watsonx.ai credentials not found. Set WX_API_KEY and WX_PROJECT_ID in .env"
    )

llm = ChatWatsonx(
    model_id="ibm/granite-4-h-small",
    url="https://us-south.ml.cloud.ibm.com",
    project_id=WX_PROJECT_ID,
    apikey=WX_API_KEY,
    params={"temperature": 0.1, "max_tokens": 512},
)

In [ ]:
# Alternative: use a local LLM (LM Studio / Ollama)
# Uncomment and set the base URL for your local server.
#
# from langchain_openai import ChatOpenAI
# llm = ChatOpenAI(
#     model="ibm/granite-4-h-small",
#     base_url="http://localhost:1234/v1",
#     api_key="none",
# )

In [ ]:
# RAG pipeline
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import PromptTemplate

PROMPT_TEMPLATE = """You are a compliance analyst assistant.
Answer the question using only the information in the provided context.
If the answer is not in the context, say "Not found in the documents".

Context:
{context}

Question: {input}
Answer:"""

prompt = PromptTemplate.from_template(PROMPT_TEMPLATE)
qa_chain = create_stuff_documents_chain(llm=llm, prompt=prompt)
rag_chain = create_retrieval_chain(retriever, qa_chain)


def ask(question: str) -> str:
    resp = rag_chain.invoke({"input": question})
    print(f"Q: {question}")
    print(f"A: {resp['answer']}\n")
    return resp["answer"]

In [ ]:
# Try it out
ask("What is the total risk exposure reported by Meridian Financial Group?")
ask("What are the main regulatory obligations described in these documents?")
ask("Were any material findings raised in the audit?")

---
## Section 7 — Information Extraction

### !!! ALTERNATIVE: enrich agent from docling-agent on DoclingDocument for entities !!!

`DocumentExtractor` pulls **specific typed fields** out of a document according to
a Pydantic schema — without converting the whole document.

> **Beta:** The extraction API is currently experimental and may change without prior notice.

In [ ]:
# For DocumentExtractor we need the full docling package (includes VLM support)
! uv pip install -q "docling[vlm]"

In [ ]:
from docling.datamodel.base_models import ConversionStatus, InputFormat
from docling.document_extractor import DocumentExtractor
from pydantic import BaseModel, Field
from rich import print as rprint


class ComplianceForm(BaseModel):
    """Compliance Assessment Form — Meridian Financial Group."""

    entity_name: str | None = Field(
        default=None,
        description="Legal name of the reporting entity",
        examples=["Meridian Financial Group"],
    )
    reporting_period: str | None = Field(
        default=None,
        description="Quarter and year covered by this report",
        examples=["Q3 2025"],
    )
    total_risk_exposure_usd: float | None = Field(
        default=None,
        description="Total risk exposure in USD millions",
    )
    highest_risk_category: str | None = Field(
        default=None,
        description="The single highest-ranked risk category",
        examples=["Credit Risk", "Market Risk", "Operational Risk"],
    )
    regulator_reference_number: str | None = Field(
        default=None,
        description="Regulatory filing or reference number",
        examples=["FRB-2025-0847"],
    )
    key_obligations: list[str] | None = Field(
        default_factory=list,
        description="List of key regulatory obligations mentioned",
    )
    material_findings: list[str] | None = Field(
        default_factory=list,
        description="Material findings raised in audit or review",
    )
    sign_off_date: list[str] | None = Field(
        default=None,
        description="Date the document was signed off (YYYY-MM-DD)",
        examples=["2025-10-15"],
    )


print("Schema defined.")

In [ ]:
# Extract from the primary PDF
extractor = DocumentExtractor(
    allowed_formats=[InputFormat.PDF, InputFormat.DOCX, InputFormat.IMAGE]
)

ext_result = extractor.extract(
    source=SOURCES[0],
    template=ComplianceForm,
)

if ext_result.status in (ConversionStatus.SUCCESS, ConversionStatus.PARTIAL_SUCCESS):
    for page in ext_result.pages:
        print(f"--- Page {page.page_no} ---")
        rprint(page.extracted_data)
else:
    print(f"Extraction failed: {ext_result.errors}")

In [ ]:
# Extract from all documents and merge into a single form
# Fields from later documents override earlier ones unless already set.
merged_form: dict = {}

for source in SOURCES:
    result = extractor.extract(source=source, template=ComplianceForm)
    if result.status in (ConversionStatus.SUCCESS, ConversionStatus.PARTIAL_SUCCESS):
        for page in result.pages:
            for field, value in page.extracted_data.items():
                if value and field not in merged_form:
                    merged_form[field] = value
                elif value and isinstance(value, list):
                    # Merge lists (e.g. key_obligations)
                    existing = merged_form.get(field, [])
                    merged_form[field] = list(dict.fromkeys(existing + value))
    else:
        print(f"  Skipped {source}: {result.errors}")

print("\n=== Merged Compliance Form ===")
rprint(merged_form)

---
## Section 8 — Putting It All Together

We now combine the extracted fields with an LLM-generated narrative summary
and render the completed Compliance Assessment Form as an HTML report.

In [ ]:
# Generate a narrative compliance summary using watsonx.ai
import json

summary_prompt = f"""You are a senior compliance analyst at Meridian Financial Group.
Based on the following extracted compliance data, write a concise 3-sentence executive
summary of the Q3 compliance posture. Focus on total risk exposure, the highest risk
category, and the most critical regulatory obligations.

Extracted data:
{json.dumps(merged_form, indent=2)}

Executive summary:"""

summary_response = llm.invoke(summary_prompt)
compliance_summary = summary_response.content
print(compliance_summary)

In [ ]:
# Render the completed form as an HTML report
from IPython.display import HTML, display


def render_compliance_form(form: dict, summary: str) -> str:
    rows = ""
    field_labels = {
        "entity_name": "Entity Name",
        "reporting_period": "Reporting Period",
        "total_risk_exposure_usd": "Total Risk Exposure (USD M)",
        "highest_risk_category": "Highest Risk Category",
        "regulator_reference_number": "Regulatory Reference No.",
        "key_obligations": "Key Obligations",
        "material_findings": "Material Findings",
        "sign_off_date": "Sign-off Date",
    }
    for field, label in field_labels.items():
        value = form.get(field)
        if isinstance(value, list):
            value_html = "<ul>" + "".join(f"<li>{v}</li>" for v in value) + "</ul>" if value else "<em>Not found</em>"
        elif value is None:
            value_html = "<em style='color:#999'>Not found</em>"
        else:
            value_html = str(value)
        rows += f"<tr><td style='font-weight:600;padding:6px 12px;'>{label}</td><td style='padding:6px 12px'>{value_html}</td></tr>"

    return f"""
<div style="font-family:system-ui,sans-serif;max-width:720px;margin:0 auto">
  <h2 style="border-bottom:2px solid #0062ff;padding-bottom:8px">Compliance Assessment Form</h2>
  <p style="color:#555"><em>Meridian Financial Group — auto-populated by Docling for IBM watsonx</em></p>
  <table style="border-collapse:collapse;width:100%;margin:16px 0">
    <thead><tr style="background:#f0f0f0">
      <th style="padding:6px 12px;text-align:left">Field</th>
      <th style="padding:6px 12px;text-align:left">Value</th>
    </tr></thead>
    <tbody>{rows}</tbody>
  </table>
  <h3>Executive Summary</h3>
  <blockquote style="border-left:4px solid #0062ff;padding:8px 16px;background:#f5f9ff;margin:0">
    {summary}
  </blockquote>
</div>
"""

html_report = render_compliance_form(merged_form, compliance_summary)
display(HTML(html_report))

In [ ]:
# Save the report to disk
report_path = output_dir / "compliance_assessment_form_filled.html"
report_path.write_text(
    f"<!DOCTYPE html><html><head><meta charset='utf-8'><title>Compliance Assessment</title></head>"
    f"<body>{html_report}</body></html>",
    encoding="utf-8",
)
print(f"Report saved to: {report_path}")

---
## Section 9 — (Optional) Edit the Form with Docling Agent

`DoclingEditingAgent` from [`docling-agent`](https://pypi.org/project/docling-agent/) applies
natural-language edits to an existing `DoclingDocument` — for example, populating a form template
with extracted values.

**Backend options:** `docling-agent` supports `ollama`, `lmstudio`, `litellm`, and `mellea`.
To use **watsonx.ai**, route through a [LiteLLM proxy](https://docs.litellm.ai/docs/) that has
the `ibm_watsonx_ai` provider configured, then pass `type="litellm"` with a
`watsonx/<model-id>` model name. Alternatively, use a local model via `ollama` for offline
experimentation — no proxy needed.

In [ ]:
# Install docling-agent (published on PyPI)
! uv pip install -q docling-agent

In [ ]:
import json
from pathlib import Path

from docling_agent.agents import (
    BackendConfig,
    DoclingEditingAgent,
    ModelConfig,
    create_backend,
)
from docling_core.types.doc.document import DoclingDocument

# ── Option A: Ollama (local, no proxy needed) ────────────────────────────────
# backend = create_backend(
#     BackendConfig(
#         type="ollama",
#         base_url="http://localhost:11434",
#         models=ModelConfig(reasoning="granite3.3:8b", writing="granite3.3:8b"),
#     )
# )

# ── Option B: watsonx.ai via a LiteLLM proxy ─────────────────────────────────
# Start LiteLLM proxy with ibm_watsonx_ai provider, then:
backend = create_backend(
    BackendConfig(
        type="litellm",
        base_url="http://localhost:4000/v1",  # LiteLLM proxy URL
        api_key_env="LITELLM_API_KEY",        # env var holding the proxy key
        models=ModelConfig(
            reasoning="watsonx/ibm/granite-4-h-small",
            writing="watsonx/ibm/granite-4-h-small",
        ),
    )
)

# Convert the primary PDF to a DoclingDocument first
# (reuses the `client` and `primary_doc` from Section 2/3)
template_doc = primary_doc   # the converted meridian_risk_disclosure DoclingDocument

agent = DoclingEditingAgent(backend=backend, tools=[])

filled = agent.run(
    task=(
        "Add a new section titled 'Compliance Assessment Summary' at the end of the document. "
        "Under it, create a two-column table with the following field–value pairs: "
        + json.dumps(merged_form, indent=2)
    ),
    document=template_doc,
)
filled.save_as_html(output_dir / "compliance_assessment_agent_filled.html")
print(f"Saved to {output_dir / 'compliance_assessment_agent_filled.html'}")

---
## Summary

In this notebook you:

| Step | What you did | Docling component |
|------|-------------|------------------|
| 1 | Connected to the managed service | `DoclingServiceClient` |
| 2 | Converted PDF, DOCX, and EPUB concurrently | `client.convert_all()` |
| 3 | Explored document structure in Python | `doc.iterate_items()`, `doc.tables` |
| 4 | Exported to DocLang and ran XPath queries | `dclq` |
| 5 | Chunked a document for RAG | `client.chunk(ChunkerKind.HYBRID)` |
| 6 | Built a RAG pipeline with LangChain + Milvus | `DoclingLoader`, `ChatWatsonx` |
| 7 | Extracted typed fields across all documents | `DocumentExtractor`, Pydantic |
| 8 | Generated an HTML compliance report | LLM summary + HTML render |
| 9 | *(Optional)* Edited a document with an agent | `DoclingEditingAgent` |

**Next steps:**
- Try the IBM Bob track (Part 2A) to do the same workflow conversationally
- Explore the [Docling documentation](https://docling-project.github.io/docling/) for advanced pipeline options
- Check the [Docling Agent examples](https://github.com/docling-project/docling-agent/tree/main/examples) for chunkless RAG, document writing, and batch enrichment